# Phase 4 — Time-Aware Train/Validation Split
## Transaction Fraud Risk Engine

**Objective:** Split the Phase 2 engineered feature dataset into training and validation sets based on chronological order (`TransactionDT`), rather than a random split. This mirrors how the model will actually be used in production — trained on past transactions, evaluated on future ones it has never seen — and avoids the data leakage that a random split would introduce on time-ordered data.

**Input data:** `data/processed/engineered_features.csv` — the Phase 2 output (590,540 rows, 14 columns), statistically validated in Phase 3.

**Structure of this notebook:** Sort by time → determine the split cutoff → verify zero time overlap between sets → save the resulting train/validation sets for use in Phase 5.

## 4.1 Chronological Split

Sort all transactions by `TransactionDT` and split into training (earliest 80%) and test (most recent 20%) sets, using a strict time cutoff rather than a random split. Then resplit that 80% into 64% training and 16% validation. This ensures the validation set only ever contains transactions that occurred after every transaction in the training set — mirroring how the model would actually be deployed and evaluated in production.

In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/engineered_features.csv")
df = df.sort_values("TransactionDT").reset_index(drop=True)

# First cut: 80% dev / 20% test
split_idx_1 = int(len(df) * 0.80)
test_threshold_dt = df.loc[split_idx_1, "TransactionDT"]

dev_df = df[df["TransactionDT"] <= test_threshold_dt].reset_index(drop=True)
test_df = df[df["TransactionDT"] > test_threshold_dt].reset_index(drop=True)

# Second cut: within dev, 80% train / 20% validation
split_idx_2 = int(len(dev_df) * 0.80)
val_threshold_dt = dev_df.loc[split_idx_2, "TransactionDT"]

train_final_df = dev_df[dev_df["TransactionDT"] <= val_threshold_dt].reset_index(drop=True)
val_df = dev_df[dev_df["TransactionDT"] > val_threshold_dt].reset_index(drop=True)

print(train_final_df.shape, val_df.shape, test_df.shape)
print("Check 1:", train_final_df["TransactionDT"].max(), "<", val_df["TransactionDT"].min())
print("Check 2:", val_df["TransactionDT"].max(), "<", test_df["TransactionDT"].min())

train_final_df.to_csv("../data/processed/train_final_v2.csv", index=False)
val_df.to_csv("../data/processed/val_set_v2.csv", index=False)
test_df.to_csv("../data/processed/test_set_v2.csv", index=False)

(377947, 79) (94486, 79) (118107, 79)
Check 1: 9466289 < 9466302
Check 2: 12192900 < 12192911


### Observations

| Column | Train Missing | Train % | Validation Missing | Validation % |
|---|---:|---:|---:|---:|
| `card4` | 828 | 0.22% | 5 | 0.005% |
| `card6` | 824 | 0.22% | 4 | 0.004% |
| `addr1` | 42,004 | 11.11% | 11,758 | 12.44% |
| `P_emaildomain` | 59,927 | 15.85% | 13,659 | 14.46% |
| `card_avg_amt_so_far` | 11,945 | 3.16% | 785 | 0.83% |
| `amt_deviation_ratio` | 11,945 | 3.16% | 785 | 0.83% |
| V-columns | 12 each | 0.003% | 97 each | 0.103% |

- **`card4` / `card6`** — very low missingness in both training and validation. These categorical variables can reasonably be handled using a dedicated `"Missing"` category or training-set mode.

- **`addr1` / `P_emaildomain`** — substantially higher missingness, ranging from approximately 11–16%. Dropping rows would unnecessarily discard a significant amount of data, so deliberate categorical missing-value handling is more appropriate.

- **`card_avg_amt_so_far` / `amt_deviation_ratio`** — these engineered features have missing values because the transaction has no previous card history from which the required historical statistics can be calculated. Therefore, the missingness has meaningful information and should not be treated as ordinary random missingness.

- **V-columns** — only a very small number of values are missing. The missingness is limited compared with the overall dataset, but the affected V-features should still be handled consistently during preprocessing.

- **Validation data will be transformed using parameters learned from the training set only.** Missing-value statistics from validation or test data will not be used to determine the imputation strategy or imputation values.

- **Conclusion:** the missing-value patterns differ substantially between features. Categorical variables require categorical missing-value treatment, while the engineered historical features require a strategy that respects the meaning of `"no prior history"`. All imputation parameters will be learned exclusively from the training set and then applied unchanged to validation and the untouched test set.

## 4.2 Phase 4 Saving and Summary

**Objective recap:** split the Phase 2 engineered feature dataset into training, validation, and final test sets using strict chronological cutoffs. This preserves the time-ordered structure of the fraud dataset and prevents future transaction periods from being used during model development.

**Result:**

* **Training set:** approximately 64% of the full dataset — the earliest portion of the development data, saved to `data/processed/train_final_v2.csv`.

* **Validation set:** approximately 16% of the full dataset — the later portion of the development data, saved to `data/processed/val_set_v2.csv`.

* **Final test set:** approximately 20% of the full dataset — the latest transaction period, saved to `data/processed/test_set_v2.csv`. This set will remain completely untouched during model development.

* **Leakage checks:** confirmed strict chronological separation:

  * Training maximum `TransactionDT` < Validation minimum `TransactionDT`
  * Validation maximum `TransactionDT` < Test minimum `TransactionDT`

**Why this matters:** this three-way time-aware split creates a proper development and evaluation workflow. Training data is used to fit models, validation data is used to make modeling decisions such as feature experiments, hyperparameter tuning, and sampling strategies, while the final test set is reserved exclusively for the final evaluation after the model and all modeling decisions have been finalized.

This ensures that the final test performance provides a much more honest estimate of how the model may perform on genuinely future transactions.


In [3]:
dev_df.to_csv("../data/processed/train_set.csv", index=False)
test_df.to_csv("../data/processed/test_set.csv", index=False)

print("Train set saved:", dev_df.shape)
print("Test set saved:", test_df.shape)

Train set saved: (472433, 79)
Test set saved: (118107, 79)


**Phase 4 status:** ✅ Complete.